# IAAIS Chapter 2 — Search Engine

This notebook demonstrates the first executable IAAIS module. The Search Engine is a general problem-solving component: a caller supplies a start state, a goal test, and an action function. The engine returns a path, cost, status, and search statistics.

## Design decision

The IAAIS domain is a finite workout-recording interpretation problem. The initial MVP design assumption is a branching factor of roughly 2–4 candidate actions per state and a solution depth of roughly 5–30 interpretation actions. Those are design assumptions, not measured results; the engine records expansion and frontier statistics so they can be checked against real sessions. The default is therefore bounded A* search. A zero heuristic makes the same implementation uniform-cost search; an admissible domain heuristic can reduce expansions while preserving least-cost guarantees.

This is not an adversarial problem: IAAIS has no opponent selecting actions against it. Minimax and MCTS are therefore not needed for this module.

The search engine does not own exercise facts. Future Knowledge Base and Planner modules will provide the domain-specific actions and heuristic through the same interface.

In [ ]:
from iaais.search_engine import (
    SearchAction,
    SearchAlgorithm,
    SearchEngine,
    SearchProblem,
)

## A small workout-interpretation state space

The states below are deliberately simple. They represent an interpretation episode in which one sensor segment can be explained either as a supported back squat or as an uncertain alternative. The uncertain path remains possible but has a review penalty. This demonstrates how later Knowledge Base evidence can become search cost without being hard-coded into the engine.

In [ ]:
problem_graph = {
    "start": [
        SearchAction(
            action="propose-back-squat",
            next_state="review-ready",
            cost=0.10,
            metadata={"confidence": 0.91, "status": "proposed"},
        ),
        SearchAction(
            action="route-to-uncertain-review",
            next_state="review-ready",
            cost=1.00,
            metadata={"confidence": 0.54, "status": "review-required"},
        ),
    ],
    "review-ready": [],
}

problem = SearchProblem(
    initial_state="start",
    goal_test=lambda state: state == "review-ready",
    actions=lambda state: problem_graph[state],
    # A lower bound: at least 0.10 cost remains from the start.
    heuristic=lambda state: 0.10 if state == "start" else 0.0,
)

In [ ]:
engine = SearchEngine(
    algorithm=SearchAlgorithm.ASTAR,
    max_expansions=100,
    heuristic_is_admissible=True,
)
result = engine.search(problem)

print("status:", result.status.value)
print("actions:", [step.action for step in result.path])
print("states:", result.states)
print("total cost:", result.total_cost)
print("optimality guaranteed:", result.optimality_guaranteed)
print("expanded nodes:", result.expanded_nodes)
print("path metadata:", [dict(step.metadata) for step in result.path])

## What this proves—and what it does not

This demonstrates a working general search component and a clean future integration seam. It does not yet prove that IAAIS can recognize real exercises, count repetitions, or infer sets. Those capabilities belong to later perception, recognition, Knowledge Base, and Planner modules. The next integration step is to replace the fixture action function with a Knowledge Base adapter while preserving this engine's public interface.